In [1]:
from sixgill.pipesim import Model
from sixgill.definitions import *
import pandas as pd
import numpy as np
import time, datetime
import json
from tqdm import tqdm

In [2]:
model=Model.open("Petani Injection Final Model-GS-Network Integrated.pips") #input your pipesim filename
model2=Model.open("Dummy Choke.pips")

INFO:manta.server.manager:Starting PIPESIM server on thread ID: 14264
INFO:manta.server.manager:Waiting for PIPESIM server to start.
INFO:manta.server.manager:Python toolkit license is checked out successfully
INFO:manta.server.manager:Waiting for PIPESIM server to start.
INFO:manta.server.manager:Starting PIPESIM server on thread ID: 43260
INFO:manta.server.manager:Python toolkit license is checked out successfully
INFO:manta.server.manager:Waiting for PIPESIM server to start.
INFO:sixgill.core.metadata:Using cached metadata for 'http://localhost:51429/api/metadata#'


In [3]:
chokedata=pd.read_excel("ChokeMatchingData.xlsx") #needed columns: WellName, UpstreamPressure, DownstreamPressure, and LiquidRate; opsional beansize
chokedata

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [43]:
well_conn=model.get_connections(Sink=ALL)
chk_params=model.get_values(Choke=ALL, parameters=["BeanSize","DischargeCoefficient",'UpstreamPipeID'])
chk_params2=model2.get_values(Choke=ALL, parameters=["BeanSize","DischargeCoefficient",'UpstreamPipeID'])
chk2=model2.find(Choke=ALL)
src2=model2.find(Source=ALL)
fl_params=model.get_values(Flowline=ALL, parameters=["InnerDiameter"])

## Input Matching Method (Options: DischargeCoefficient and BeanSize) 

In [65]:
matching_method = input()
print("Matching Method:", matching_method) #BeanSize, or DischargeCoefficient

BeanSize
Matching Method: BeanSize


In [66]:
matching_method

'BeanSize'

## Input default value for DischargeCoefficient 

In [61]:
cd_default = input()
print("Default Cd:", cd_default)

0.65
Default Cd: 0.65


In [63]:
cd_default=float(cd_default)

In [64]:
cd_default

0.65

In [47]:
chk_params2

defaultdict(dict,
            {'Ck': {'BeanSize': 1.273133,
              'DischargeCoefficient': 0.6,
              'UpstreamPipeID': 4.0}})

In [48]:
chk_conn=model.get_connections(Choke=ALL)
cvl_conn=model.get_connections(CheckValve=ALL)
junc_conn=model.get_connections(Junction=ALL)
flowline_conn=model.get_connections(Flowline=ALL)
wells=model.find(Sink=ALL)

In [49]:
chk_conn

{'CHK_BEKASAP-0010-01': {'Source': 'Cvl_BEKASAP-0010-01',
  'Source Port': '',
  'Destination': 'BEKASAP-0010-01',
  'Top Port': '',
  'Bottom Port': '',
  'Boot Port': ''},
 'CHK_BEKASAP-0011-01': {'Source': 'Cvl_BEKASAP-0011-01',
  'Source Port': '',
  'Destination': 'BEKASAP-0011-01',
  'Top Port': '',
  'Bottom Port': '',
  'Boot Port': ''},
 'CHK_BEKASAP-0032-01': {'Source': 'Cvl_BEKASAP-0032-01',
  'Source Port': '',
  'Destination': 'BEKASAP-0032-01',
  'Top Port': '',
  'Bottom Port': '',
  'Boot Port': ''},
 'CHK_BEKASAP-0036-01': {'Source': 'Cvl_BEKASAP-0036-01',
  'Source Port': '',
  'Destination': 'BEKASAP-0036-01',
  'Top Port': '',
  'Bottom Port': '',
  'Boot Port': ''},
 'CHK_BEKASAP-0039-01': {'Source': 'Cvl_BEKASAP-0039-01',
  'Source Port': '',
  'Destination': 'BEKASAP-0039-01',
  'Top Port': '',
  'Bottom Port': '',
  'Boot Port': ''},
 'CHK_BEKASAP-0044-01': {'Source': 'Cvl_BEKASAP-0044-01',
  'Source Port': '',
  'Destination': 'BEKASAP-0044-01',
  'Top Port': '

In [50]:
def fl_chk_conn(chk, choke_conn, junc_conn, cvl_conn, flowline_conn,wells):
    fl_chk= {}
    j = choke_conn[chk]["Source"]
    if j in wells:
        j = choke_conn[chk]["Destination"]
    usejunc = True
    if j in junc_conn.keys():
        usejunc = True
    elif j in cvl_conn.keys():
        usejunc = False
    if usejunc:
        if junc_conn[j]["Source"] != chk and junc_conn[j]["Source"] != "":
            fl_chk[chk] = junc_conn[j]["Source"]
        elif (
            junc_conn[j]["Destination"] != chk
            and junc_conn[j]["Destination"] != ""
        ):
            fl_chk[chk] = junc_conn[j]["Destination"]
        else:
            for fl in flowline_conn:
                if flowline_conn[fl]["Source"] == j:
                    fl_chk[chk] = fl
                elif flowline_conn[fl]["Destination"] == j:
                    fl_chk[chk] = fl
    else:
        if cvl_conn[j]["Source"] != chk and cvl_conn[j]["Source"] != "":
            fl_chk[chk] = cvl_conn[j]["Source"]
        elif (
            cvl_conn[j]["Destination"] != chk
            and cvl_conn[j]["Destination"] != ""
        ):
            fl_chk[chk] = cvl_conn[j]["Destination"]
        else:
            for fl in flowline_conn:
                if flowline_conn[fl]["Source"] == j:
                    fl_chk[chk] = fl
                elif flowline_conn[fl]["Destination"] == j:
                    fl_chk[chk] = fl
    return fl_chk[chk]

In [ ]:
matched={}
matched={}
simulation={}
for wl in tqdm(chokedata["WellName"]):
    if chokedata[chokedata["WellName"]==wl]["UpstreamPressure"].values[0]-chokedata[chokedata["WellName"]==wl]["DownstreamPressure"].values[0]==0:
        print(wl, " is not matched because deltapchoke is 0")
        matched[wl]={}
        matched[wl][matching_method]=float("nan")
        continue
    matched[wl]={}
    up=chokedata[chokedata["WellName"]==wl]["UpstreamPressure"].values[0]
    down=chokedata[chokedata["WellName"]==wl]["DownstreamPressure"].values[0]
    liqrate=chokedata[chokedata["WellName"]==wl]["LiquidRate"].values[0]
    if well_conn[wl]["Source"]!='':
        chk=well_conn[wl]["Source"]
    else:
        chk=well_conn[wl]["Destination"]
    fl=fl_chk_conn(chk, chk_conn, junc_conn, cvl_conn, flowline_conn, wells)
    cd=chk_params[chk]["DischargeCoefficient"]
    beansize=chk_params[chk]["BeanSize"]
    if matching_method=="DischargeCoefficient":
        minval=0
        maxval=5
    else:
        minval=0.1
        maxval=6
    print(wl, fl, chk)
    model.set_value(context=chk, parameter='IsActive', value=True)
    model.set_value(context=chk, parameter='CriticalPressureRatio', value=0.1)
    model.set_value(context=chk, component="Choke", parameter='UpstreamPipeID', value=fl_params_ori[fl]["InnerDiameter"])
    model2.set_value(context=chk2[0], component="Choke", parameter='UpstreamPipeID', value=fl_params[fl]["InnerDiameter"])
    model2.set_value(context=chk2[0], parameter='DischargeCoefficient', value=cd_default)
    model2.set_value(context=chk2[0], parameter='BeanSize', value=beansize)
    matched[wl]["InjectionRate"]=liqrate
    matched[wl]["UpstreamID"]=fl_params[fl]["InnerDiameter"]
    if matching_method=='DischargeCoefficient':
        matched[wl]["BeanSize"]=beansize
    else:
        matched[wl]["DischargeCoefficient"]=cd_default
    matched[wl]["UpstreamPressure"]=up
    matched[wl]["DownstreamPressure"]=down
    matched[wl]["DeltaPChoke"]=up-down
    print(model2.get_values(Choke=ALL, parameters=["BeanSize","DischargeCoefficient",'UpstreamPipeID']))
    print(wl, ' Injection Rate: ', liqrate)
    print(fl,' Inner Diameter: ', fl_params[fl]["InnerDiameter"])
    print(chk, ' Beansize: ', beansize, ' Cd: ', cd, ' DP: ', up-down, ' UpstreamP: ', up, ' DownstreamP: ', down)
    simulation[wl]=model2.tasks.ptprofilesimulation.run(
                    src2[0],
                    parameters={
                        Parameters.PTProfileSimulation.OUTLETPRESSURE:down,
                        Parameters.PTProfileSimulation.INLETPRESSURE:up,
                        Parameters.PTProfileSimulation.CALCULATEDVARIABLE:'calcCustom',
                        Parameters.PTProfileSimulation.LIQUIDFLOWRATE:liqrate,
                        Parameters.PTProfileSimulation.CUSTOMVARIABLE:{
                            Parameters.PTProfileSimulation.CustomVariable.COMPONENT:'Ck',
                            Parameters.PTProfileSimulation.CustomVariable.VARIABLE:matching_method,
                            Parameters.PTProfileSimulation.CustomVariable.MINVALUE:minval,
                            Parameters.PTProfileSimulation.CustomVariable.MAXVALUE:maxval,
                            Parameters.PTProfileSimulation.CustomVariable.ISDIRECTPROPORTIONALITY:True
                        }
                    },
                    profile_variables=['Pressure'],
                )
    
    print(simulation[wl].summary)
    print(simulation[wl].cases)
    if "UNCONVERGED" in simulation[wl].cases[0]:
        simulation[wl]=model2.tasks.ptprofilesimulation.run(
                    src2[0],
                    parameters={
                        Parameters.PTProfileSimulation.OUTLETPRESSURE:down,
                        Parameters.PTProfileSimulation.INLETPRESSURE:up,
                        Parameters.PTProfileSimulation.CALCULATEDVARIABLE:'calcCustom',
                        Parameters.PTProfileSimulation.LIQUIDFLOWRATE:liqrate,
                        Parameters.PTProfileSimulation.CUSTOMVARIABLE:{
                            Parameters.PTProfileSimulation.CustomVariable.COMPONENT:'Ck',
                            Parameters.PTProfileSimulation.CustomVariable.VARIABLE:matching_method,
                            Parameters.PTProfileSimulation.CustomVariable.MINVALUE:minval,
                            Parameters.PTProfileSimulation.CustomVariable.MAXVALUE:maxval,
                            Parameters.PTProfileSimulation.CustomVariable.ISDIRECTPROPORTIONALITY:False
                        }
                    },
                    profile_variables=['Pressure'],
                )
        print(simulation[wl].summary)
        print(simulation[wl].cases)
        if "UNCONVERGED" in simulation[wl].cases[0]:
            matched[wl][matching_method]="Unconverged"
            print(wl, ' has no solution')
            continue
    try:
        a=simulation[wl].cases[0].split('=')[-1]
        matched[wl][matching_method]=float(a.split(' ')[0])
        print(wl, matching_method,' : ',float(a.split(' ')[0]))
        chk_params[chk][matching_method]=matched[wl][matching_method]
        model.set_values(chk_params)
        try:
            model.save()
        except:
            model.save()
        print(wl, ' has solution by changing '+matching_method+' .')
    except IndexError:
        print(wl, ' has no solution')

  0%|                                                                                           | 0/38 [00:00<?, ?it/s]

AMAN-0004-01  is not matched because deltapchoke is 0
AMAN-0007-01  is not matched because deltapchoke is 0
AMAN-0009-01  is not matched because deltapchoke is 0
AMAN-0010-01  is not matched because deltapchoke is 0
BEKASAP-0003-01  is not matched because deltapchoke is 0
BEKASAP-0004-01  is not matched because deltapchoke is 0
BEKASAP-0010-01 WIL-BEKA-10_MN CHK_BEKASAP-0010-01
defaultdict(<class 'dict'>, {'Ck': {'BeanSize': 1.2731329859837897, 'DischargeCoefficient': 0.65, 'UpstreamPipeID': 4.026}})
BEKASAP-0010-01  Injection Rate:  15923
WIL-BEKA-10_MN  Inner Diameter:  4.026
CHK_BEKASAP-0010-01  Beansize:  1.2731329859837897  Cd:  1.029799  DP:  90  UpstreamP:  570  DownstreamP:  480


INFO:sixgill.core.run_or_start_simulation:Simulation is running.
INFO:sixgill.core.run_or_start_simulation:Simulation is running.
INFO:sixgill.core.run_or_start_simulation:Simulation is finished.
INFO:sixgill.core.run_or_start_simulation:Simulation run successfully.


{'Info': ['2025-07-22T11:54:26.307Z    Beginning simulation...', '2025-07-22T11:54:26.307Z    Running with 1 core', '2025-07-22T11:54:28.229Z    Solver converged to a solution.', '2025-07-22T11:54:28.271Z    Finished simulation'], 'Error': [], 'Warning': []}
['Ck-BeanSize=1.597067 ins']
BEKASAP-0010-01 BeanSize  :  1.597067


 18%|███████████████▎                                                                   | 7/38 [00:09<00:40,  1.31s/it]

...

INFO:sixgill.core.run_or_start_simulation:Simulation is running.
INFO:sixgill.core.run_or_start_simulation:Simulation is running.
INFO:sixgill.core.run_or_start_simulation:Simulation is finished.
INFO:sixgill.core.run_or_start_simulation:Simulation run successfully.


{'Info': ['2025-07-22T11:55:59.974Z    Beginning simulation...', '2025-07-22T11:55:59.974Z    Running with 1 core', '2025-07-22T11:56:01.956Z    Solver converged to a solution.', '2025-07-22T11:56:01.995Z    Finished simulation'], 'Error': [], 'Warning': []}
['Ck-BeanSize=1.027331 ins']
BEKASAP-0096-01 BeanSize  :  1.027331


 84%|█████████████████████████████████████████████████████████████████████             | 32/38 [01:42<00:20,  3.39s/it]

BEKASAP-0096-01  has solution by changing BeanSize .
BEKASAP-0098-01 WIL-BEKA-98-2 CHK_BEKASAP-0098-01
defaultdict(<class 'dict'>, {'Ck': {'BeanSize': 4.0, 'DischargeCoefficient': 0.65, 'UpstreamPipeID': 4.026}})
BEKASAP-0098-01  Injection Rate:  19546
WIL-BEKA-98-2  Inner Diameter:  4.026
CHK_BEKASAP-0098-01  Beansize:  4.0  Cd:  0.0434685  DP:  20  UpstreamP:  660  DownstreamP:  640


INFO:sixgill.core.run_or_start_simulation:Simulation is running.


In [ ]:
result=pd.DataFrame.from_dict(matched, orient='index')
result

In [ ]:
result.to_excel("Delta Pressure Choke Matching Result Changing"+ matching_method+".xlsx")